In [3]:
import numpy as np

def p(x, theta):
    return (theta - 1) / (x ** theta) if x >= 1 else 0

def F(x, theta):
    return (1 - x ** (1 - theta)) if x >= 1 else 0

def inverse_f(y, theta):
    return (1 - y) ** (1 / (1 - theta))

# Параметры
n = 100
beta = 0.95
theta_true = 10000
np.random.seed(42)

# Генерация выборки
random_numbers = np.random.random(size=n)
sample = np.array([inverse_f(y, theta_true) for y in random_numbers])
ln_mean = np.mean([np.log(x) for x in sample])

confidence_intervals = {}

# Вывод выборки
print("=" * 80)
print(" " * 35 + "Выборка")
print("=" * 80)

col_width = 18
header = f"{'Значение 1':>{col_width}} │ {'Значение 2':>{col_width}} │ {'Значение 3':>{col_width}} │ {'Значение 4':>{col_width}}"
print(header)
print("─" * len(header))

for i in range(0, len(sample), 4):
    row = sample[i:i+4]
    cells = []
    for j in range(4):
        if j < len(row):
            cells.append(f"{row[j]:>{col_width}.7f}")
        else:
            cells.append(" " * col_width)
    print(" │ ".join(cells))

print("=" * 80)
print()

# Асимптотический доверительный интервал (О.М.П.) для медианы
median_lower = 2 ** ln_mean - (1.96 * np.log(2) * 2 ** ln_mean * ln_mean) / np.sqrt(n)
median_upper = 2 ** ln_mean + (1.96 * np.log(2) * 2 ** ln_mean * ln_mean) / np.sqrt(n)

print("=" * 70)
print(" " * 15 + "Асимптотический доверительный интервал (О.М.П.) для медианы")
print("=" * 70)
print(f"   {median_lower:.6f}  <  x₀₅  <  {median_upper:.6f}")
print(f"   Длина интервала: {median_upper - median_lower:.6f}")
print(f"   Выборочная медиана: {np.median(sample):.6f}")
print("=" * 70)
print()

# Асимптотический доверительный интервал (О.М.П.)
theta_lower = 1 - (1.96 - np.sqrt(n)) / (ln_mean * np.sqrt(n))
theta_upper = 1 + (1.96 + np.sqrt(n)) / (ln_mean * np.sqrt(n))

print("=" * 60)
print(" " * 18 + "Асимптотический доверительный интервал (О.М.П.)")
print("=" * 60)
print(f"   {theta_lower:.6f}  <  θ  <  {theta_upper:.6f}")
print(f"   Длина интервала: {theta_upper - theta_lower:.6f}")
print("=" * 60)
print()

confidence_intervals["Асимптотический доверительный интервал (О.М.П.)"] = theta_upper - theta_lower

# Непараметрический bootstrap
print("Непараметрический bootstrap:")
bootstrap_iterations = 1000
theta_omp = 1 + 1 / ln_mean.astype(float)

bootstrap_delta = []
for _ in range(bootstrap_iterations):
    boot_sample = np.random.choice(sample, size=n, replace=True)
    boot_ln_mean = np.mean([np.log(x) for x in boot_sample])
    boot_theta = 1 + 1 / boot_ln_mean
    bootstrap_delta.append(boot_theta - theta_omp)

bootstrap_delta_sorted = np.array(sorted(bootstrap_delta))
alpha = 1 - beta
lower_percentile = int((alpha / 2) * bootstrap_iterations)
upper_percentile = int((1 - alpha / 2) * bootstrap_iterations)

nonparam_lower = theta_omp - bootstrap_delta_sorted[upper_percentile]
nonparam_upper = theta_omp - bootstrap_delta_sorted[lower_percentile]

print("=" * 70)
print(" " * 17 + "Непараметрический bootstrap'овский доверительный интервал")
print("=" * 70)
print(f"   {nonparam_lower:.6f}  <  θ  <  {nonparam_upper:.6f}")
print(f"   Длина интервала: {nonparam_upper - nonparam_lower:.6f}")
print("=" * 70)
print()

confidence_intervals["Непараметрический bootstrap'овский доверительный интервал"] = nonparam_upper - nonparam_lower

# Параметрический bootstrap
print("Параметрический bootstrap:")
bootstrap_iterations = 50000

bootstrap_delta = []
for _ in range(bootstrap_iterations):
    random_sample = [inverse_f(np.random.random(), theta_omp) for _ in range(n)]
    boot_ln_mean = np.mean([np.log(x) for x in random_sample])
    boot_theta = 1 + 1 / boot_ln_mean
    bootstrap_delta.append(boot_theta - theta_omp)

bootstrap_delta_sorted = np.array(sorted(bootstrap_delta))
alpha = 1 - beta
lower_percentile = int((alpha / 2) * bootstrap_iterations)
upper_percentile = int((1 - alpha / 2) * bootstrap_iterations)

param_lower = theta_omp - bootstrap_delta_sorted[upper_percentile]
param_upper = theta_omp - bootstrap_delta_sorted[lower_percentile]

print("=" * 70)
print(" " * 18 + "Параметрический bootstrap'овский доверительный интервал")
print("=" * 70)
print(f"   {param_lower:.6f}  <  θ  <  {param_upper:.6f}")
print(f"   Длина интервала: {param_upper - param_lower:.6f}")
print("=" * 70)
print()

confidence_intervals["Параметрический bootstrap'овский доверительный интервал"] = param_upper - param_lower

# Сравнение интервалов
sorted_intervals = sorted(confidence_intervals.items(), key=lambda item: item[1])

print("=" * 70)
print(" " * 20 + "Доверительные интервалы (в порядке улучшения)")
print("=" * 70)
for interval_name, interval_value in sorted_intervals:
    print(f"   • {interval_name}  (l = {interval_value:.2f})")
print("=" * 70)

                                   Выборка
        Значение 1 │         Значение 2 │         Значение 3 │         Значение 4
─────────────────────────────────────────────────────────────────────────────────
         1.0000469 │          1.0003011 │          1.0001317 │          1.0000913
         1.0000170 │          1.0000170 │          1.0000060 │          1.0002012
         1.0000919 │          1.0001231 │          1.0000021 │          1.0003505
         1.0001787 │          1.0000239 │          1.0000201 │          1.0000203
         1.0000363 │          1.0000744 │          1.0000566 │          1.0000344
         1.0000947 │          1.0000150 │          1.0000346 │          1.0000456
         1.0000609 │          1.0001538 │          1.0000223 │          1.0000722
         1.0000898 │          1.0000048 │          1.0000935 │          1.0000187
         1.0000067 │          1.0002974 │          1.0003372 │          1.0001653
         1.0000363 │          1.0000103 │          1.00